In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os

csv_path = os.path.join(path, "Q3_data.csv")
data = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
print("First 5 rows of the dataset:")
display(data.head())

In [ ]:
# Task 3: Write your code here:
print("\nDataset Info:")
data.info()


In [ ]:
# Task 4: Write your code here:
print("\nStatistical Description:")
display(data.describe())

In [ ]:
# Task 1: Write your code here:
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

for col in data.columns:
    if data[col].dtype in ['float64', 'int64']:
        data[col].fillna(data[col].mean(), inplace=True)
    else:
        data[col].fillna(data[col].mode()[0], inplace=True)


In [ ]:
# Task 2: Write your code here:
data.drop_duplicates(inplace=True)

In [ ]:
# Task 3: Write your code here:
categorical_cols = data.select_dtypes(include=['object']).columns
if len(categorical_cols) > 0:
    for col in categorical_cols:
        le = LabelEncoder()
        data[col] = le.fit_transform(data[col])

In [ ]:
# Task 4: Write your code here:
numerical_cols = data.select_dtypes(include=['float64', 'int64']).columns.tolist()
numerical_cols.remove('B_3')
scaler = StandardScaler()
data[numerical_cols] = scaler.fit_transform(data[numerical_cols])

In [ ]:
# Task 5: Write your code here:
target_col = 'S_3'
target_counts = data[target_col].value_counts()
print(f"Target distribution:\n{target_counts}")
imbalance_ratio = target_counts.min() / target_counts.max()
print(f"Imbalance ratio (min/max): {imbalance_ratio:.2f}")
if imbalance_ratio < 0.5:
    print("The dataset is imbalanced.")
else:
    print("The dataset is reasonably balanced.")

In [ ]:
# Task 1: Write your code here:
!pip install catboost --quiet


import pandas as pd
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier
import kagglehub

avg_f1 = np.mean(f1_scores)
print(f"\nAverage F1 Score across 5 folds: {avg_f1:.4f}")

X = data.drop(columns=[target_col])
y = data[target_col]

In [ ]:
# Task 2,3,4,5: Write your code here:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

for fold, (train_index, val_index) in enumerate(kf.split(X, y), 1):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    model = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, verbose=0, random_seed=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    f1 = f1_score(y_val, y_pred)
    f1_scores.append(f1)
    print(f"Fold {fold} - F1 Score: {f1:.4f}")

avg_f1 = np.mean(f1_scores)
print(f"\nAverage F1 Score across 5 folds: {avg_f1:.4f}")




In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

feature_importances = model.get_feature_importance()
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

In [ ]:
# Task 2: Write your code here:
print("Top 10 features by importance:")
display(importance_df.head(10))

plt.figure(figsize=(10,5))
plt.barh(importance_df['Feature'][:10][::-1], importance_df['Importance'][:10][::-1])
plt.xlabel("Importance")
plt.title("Top 10 Feature Importances")
plt.show()


In [ ]:
# Task Bonus: Write your code here:
X_golden = X[[golden_feature]]

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores_golden = []

for fold, (train_index, val_index) in enumerate(kf.split(X_golden, y), 1):
    X_train, X_val = X_golden.iloc[train_index], X_golden.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    model_golden = CatBoostClassifier(
        iterations=500,
        learning_rate=0.1,
        depth=6,
        verbose=0,
        random_seed=42
    )
    model_golden.fit(X_train, y_train)

    y_pred = model_golden.predict(X_val)
    f1 = f1_score(y_val, y_pred)
    f1_scores_golden.append(f1)
    print(f"Fold {fold} - F1 Score (Golden Feature only): {f1:.4f}")

avg_f1_golden = np.mean(f1_scores_golden)
print(f"\nAverage F1 Score using only the Golden Feature: {avg_f1_golden:.4f}")

print(f"Full model Average F1: {avg_f1:.4f}")
print(f"Golden Feature only Average F1: {avg_f1_golden:.4f}")
